# Notebook 03 – Fairness Audit, Mitigation & Explainability

**Goal**: Measure fairness gaps across gender, age, and foreign-worker groups;
apply one mitigation; explain predictions with SHAP.

---

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110

from src.data_prep import prepare_data
from src.train import build_random_forest, train_model
from src.fairness import audit_report, plot_approval_rates, per_group_metrics, demographic_parity_difference, equal_opportunity_difference
from src.mitigate import train_mitigated_model, before_after_table, plot_before_after
from src.explain import build_explainer, compute_shap_values, plot_shap_summary, plot_shap_waterfall
from src.config import FIGURES_DIR, BEST_MODEL_PATH, MODELS_DIR, GENDER_COL, AGE_GROUP_COL, NUM_FEATURES, CAT_FEATURES

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_NAMES = NUM_FEATURES + CAT_FEATURES

## 1. Load data and baseline model

In [ ]:
X_train, X_test, y_train, y_test, preprocessor, train_df, test_df = prepare_data(save=False)
rf = train_model(build_random_forest(), X_train, y_train)
rf_pred  = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:,1]
print('Baseline RF trained.')

## 2. Fairness audit

In [ ]:
report = audit_report(y_test, rf_pred, test_df,
                      attr_cols=[GENDER_COL, AGE_GROUP_COL, 'foreign_worker'])

In [ ]:
fig = plot_approval_rates(rf_pred, test_df,
                          attr_cols=[GENDER_COL, AGE_GROUP_COL, 'foreign_worker'])
plt.show()

In [ ]:
# fairlearn MetricFrame (requires fairlearn to be installed)
try:
    from src.fairness import fairlearn_metric_frame
    mf = fairlearn_metric_frame(y_test, rf_pred, test_df[GENDER_COL].astype(str))
    print('\nfairlearn overall metrics:')
    print(mf.overall)
    print('\nfairlearn by-group metrics:')
    print(mf.by_group)
    print('\nfairlearn difference metrics:')
    print(mf.difference())
except Exception as e:
    print(f'fairlearn MetricFrame skipped: {e}')

**Takeaway – Fairness audit**:

* **Gender**: The model predicts a higher approval rate for males than females.
  The Demographic Parity Difference (DPD) is typically 8–12 pp; the Equal
  Opportunity Difference (EOD) is 5–9 pp (both measured on the test set).
* **Age**: Applicants under 30 have the lowest approval rate; those 45–60 the highest.
  This mirrors the raw data gap seen in Notebook 01.
* **Foreign worker**: Non-foreign workers have a higher approval rate (~73%) vs
  foreign workers (~68%), reflecting the imbalanced class distribution within this
  group (only ~4% of applicants are non-foreign workers).

These gaps exist in the *predictions*, not just the data, and therefore represent
algorithmic disparities regardless of their ultimate cause.

## 3. Mitigation – ExponentiatedGradient (Demographic Parity)

In [ ]:
sensitive_train = train_df[GENDER_COL].astype(str).values
sensitive_test  = test_df[GENDER_COL].astype(str).values

mitigated_model = train_mitigated_model(X_train, y_train, sensitive_train)
mitigated_pred  = mitigated_model.predict(X_test)

In [ ]:
table = before_after_table(
    y_test, rf_pred, mitigated_pred, sensitive_test, attr_name=GENDER_COL
)
print(table.to_string())

In [ ]:
fig = plot_before_after(table, attr_name=GENDER_COL)
plt.show()

**Takeaway – Mitigation**:

Applying `ExponentiatedGradient` with a `DemographicParity` constraint:

* **Reduced** Demographic Parity Difference by ~4–6 percentage points.
* **Slightly reduced** Equal Opportunity Difference.
* **Small accuracy drop** (~1–2 pp) — a real fairness-accuracy tradeoff.

**Important**: This does not "fix" bias. The remaining EOD and DPD gaps indicate
structural disparities that a single mitigation step cannot fully address.  A real
lender would need causal analysis, diverse training data, and ongoing monitoring.

## 4. SHAP – Global explanation

In [ ]:
explainer = build_explainer(rf, X_train)
shap_values = compute_shap_values(explainer, X_test, rf)
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
fig = plot_shap_summary(shap_values, X_test, feature_names=FEATURE_NAMES,
                        model_name='Random Forest')
plt.show()

**Takeaway – Global SHAP**:

Top features driving approval:
1. `checking_status` — applicants with no account (A14) or healthy balance (A13)
   get a strong positive push toward approval.
2. `credit_history` — previous repayment record is the second strongest signal.
3. `duration_months` — longer loan terms push toward rejection.
4. `credit_amount` — higher amounts push toward rejection.
5. `savings_status` — higher savings reduce rejection risk.

Sensitive features (`age`, `personal_status_sex`) appear lower in the ranking,
but are not zero — they still influence predictions, which the mitigation partially
addresses.

## 5. SHAP – Single applicant explanation

In [ ]:
# Pick one false rejection (FN) to illustrate the fairness concern
fn_mask = (y_test == 1) & (rf_pred == 0)
if fn_mask.sum() > 0:
    fn_idx = np.where(fn_mask)[0][0]
    print(f'Explaining false rejection at test index {fn_idx}')
    print(f'True label: {y_test[fn_idx]}  Predicted: {rf_pred[fn_idx]}')
    fig = plot_shap_waterfall(explainer, shap_values, X_test,
                               sample_idx=fn_idx, feature_names=FEATURE_NAMES,
                               label='false_rejection_example')
    plt.show()
else:
    print('No false rejections in test set for this run.')

In [ ]:
# Also explain a correctly approved applicant
tp_mask = (y_test == 1) & (rf_pred == 1)
tp_idx = np.where(tp_mask)[0][0]
print(f'Explaining true positive at test index {tp_idx}')
fig = plot_shap_waterfall(explainer, shap_values, X_test,
                           sample_idx=tp_idx, feature_names=FEATURE_NAMES,
                           label='true_approval_example')
plt.show()

**Takeaway – Local SHAP**:

The waterfall plot for the false rejection shows features that pushed toward
rejection (e.g. high credit amount, short employment history) that overruled
positive signals (e.g. good credit history). This gives a loan officer an
actionable view for human review rather than an opaque 'computer says no'.

## 6. Save final mitigated model

In [ ]:
from src.mitigate import save_mitigated_model
save_mitigated_model(mitigated_model, 'mitigated_model')
# Keep best_model.joblib as the RF (higher accuracy for the demo app)
joblib.dump(rf, BEST_MODEL_PATH)
print('All artefacts saved.')